# Comparison

Use this notebook only to orchestrate Tesseract vs Dolphin comparison runs.
Import reusable logic from `src/`; do not implement business logic here.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

IS_COLAB = 'google.colab' in sys.modules
DEFAULT_COLAB_ROOT = Path(os.environ.get("NFSE_PROJECT_ROOT", "/content/nfse-extractor"))
DEFAULT_LOCAL_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

COMPARISON_CONFIG = {
    "project_root": DEFAULT_COLAB_ROOT if IS_COLAB else DEFAULT_LOCAL_ROOT,
    "mount_drive": os.environ.get("NFSE_MOUNT_DRIVE", "0") == "1",
    "run_bootstrap": os.environ.get("NFSE_RUN_BOOTSTRAP", "0") == "1",
    "dataset_dir": Path(os.environ.get("NFSE_DATASET_DIR", "/content/nfse-samples")),
    "dataset_glob": os.environ.get("NFSE_DATASET_GLOB", "*.png"),
    "sample_limit": int(os.environ.get("NFSE_SAMPLE_LIMIT", "1")),
    "output_root": Path(os.environ.get("NFSE_COMPARISON_OUTPUT", "/content/nfse-comparison-output")),
    "experiment_name": os.environ.get("NFSE_EXPERIMENT_NAME", "colab-comparison"),
    "output_normalizer_path": os.environ.get("NFSE_OUTPUT_NORMALIZER", ""),
    "pdf_converter_factory_path": os.environ.get("NFSE_PDF_CONVERTER_FACTORY", ""),
    "dolphin_runtime_factory_path": os.environ.get("NFSE_DOLPHIN_RUNTIME_FACTORY", ""),
    "dolphin_model_path": os.environ.get("NFSE_DOLPHIN_MODEL_PATH", ""),
    "dolphin_device": os.environ.get("NFSE_DOLPHIN_DEVICE", "cpu"),
    "tesseract_language": os.environ.get("NFSE_TESSERACT_LANGUAGE", "por"),
}

if IS_COLAB and COMPARISON_CONFIG["mount_drive"]:
    from google.colab import drive
    drive.mount("/content/drive")

if IS_COLAB and COMPARISON_CONFIG["run_bootstrap"]:
    bootstrap_script = COMPARISON_CONFIG["project_root"] / "scripts" / "colab_bootstrap.sh"
    subprocess.run(["bash", str(bootstrap_script)], check=True)

if not COMPARISON_CONFIG["project_root"].exists():
    raise FileNotFoundError(f"Project root not found: {COMPARISON_CONFIG['project_root']}")

project_root_str = str(COMPARISON_CONFIG["project_root"])
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

COMPARISON_CONFIG

In [ ]:
from src.core import ExperimentComparisonRunner
from src.decision import ConfigDrivenDecisionEngine
from src.engines import DolphinExtractionAdapter, TesseractExtractionAdapter
from src.resolver import ConfigDrivenFieldResolver
from src.validation import ConfigDrivenValidator


def load_from_path(spec: str):
    module_name, attr_name = spec.split(":", 1)
    module = importlib.import_module(module_name)
    return getattr(module, attr_name)


def maybe_build(spec: str, **kwargs):
    if not spec:
        return None
    factory = load_from_path(spec)
    return factory(**kwargs) if callable(factory) else factory


if not COMPARISON_CONFIG["output_normalizer_path"]:
    raise ValueError(
        "Set NFSE_OUTPUT_NORMALIZER to a dotted path like package.module:factory before running comparison."
    )

dataset_paths = sorted(COMPARISON_CONFIG["dataset_dir"].glob(COMPARISON_CONFIG["dataset_glob"]))
dataset_paths = [path for path in dataset_paths if path.is_file()][: COMPARISON_CONFIG["sample_limit"]]
if not dataset_paths:
    raise FileNotFoundError(
        f"No dataset files matched {COMPARISON_CONFIG['dataset_glob']!r} in {COMPARISON_CONFIG['dataset_dir']}"
    )

output_normalizer_factory = load_from_path(COMPARISON_CONFIG["output_normalizer_path"])
output_normalizer = output_normalizer_factory()
pdf_converter = maybe_build(COMPARISON_CONFIG["pdf_converter_factory_path"])
dolphin_runtime_factory = maybe_build(COMPARISON_CONFIG["dolphin_runtime_factory_path"])

runner = ExperimentComparisonRunner(
    engines={
        "tesseract": TesseractExtractionAdapter(language=COMPARISON_CONFIG["tesseract_language"]),
        "dolphin": DolphinExtractionAdapter(
            runtime_factory=dolphin_runtime_factory,
            model_path=COMPARISON_CONFIG["dolphin_model_path"] or None,
            device=COMPARISON_CONFIG["dolphin_device"],
        ),
    },
    output_normalizer=output_normalizer,
    field_resolver=ConfigDrivenFieldResolver(),
    validator=ConfigDrivenValidator(),
    decision_engine=ConfigDrivenDecisionEngine(),
    output_root=COMPARISON_CONFIG["output_root"],
    pdf_converter=pdf_converter,
)

comparison_result = runner.run(
    dataset_paths,
    experiment_name=COMPARISON_CONFIG["experiment_name"],
)

comparison_result["experiment_root"]

In [ ]:
summary_path = comparison_result["experiment_summary_path"]
document_metrics_path = comparison_result["document_metrics_path"]
field_metrics_path = comparison_result["field_metrics_path"]

summary_payload = json.loads(summary_path.read_text(encoding="utf-8"))
document_metrics = json.loads(document_metrics_path.read_text(encoding="utf-8"))
field_metrics = json.loads(field_metrics_path.read_text(encoding="utf-8"))

print("Experiment Summary")
print(json.dumps(summary_payload, indent=2, ensure_ascii=False))

print("\nPer-document comparison")
for item in document_metrics:
    print(
        f"- engine={item['engine_id']} document={item['document_id']} status={item['document_status']} "
        f"fill_rate={item['fill_rate']:.3f} conflicts={item['conflict_count']} time_ms={item['processing_time_ms']:.3f}"
    )

interesting_fields = [
    item for item in field_metrics
    if item["fill_rate"] > 0 or item["conflict_rate"] > 0 or item["correctness_evaluated_count"] > 0
]

print("\nPer-field comparison")
for item in interesting_fields:
    correctness = item["correctness_rate"]
    correctness_text = "n/a" if correctness is None else f"{correctness:.3f}"
    print(
        f"- engine={item['engine_id']} field={item['field_name']} fill_rate={item['fill_rate']:.3f} "
        f"conflict_rate={item['conflict_rate']:.3f} correctness_rate={correctness_text}"
    )

print("\nArtifacts")
print(f"summary: {summary_path}")
print(f"document metrics: {document_metrics_path}")
print(f"field metrics: {field_metrics_path}")
